# Stage 02 — Feature Engineering & Windowing

Turns the cleaned OHLCV dataset into the **windowed feature datasets** the modelling notebooks consume.

Pipeline: load cleaned data → derive **stationary** base features → *(chart patterns — placeholder)* → build look-back windows + log-return targets → save `.npz` to the Shared Drive.

**Why stationary features?** The targets are log returns, so the inputs must be comparable across price levels too. The index runs ~1,270 (2006) → ~6,850 (2025); raw price levels in the test era fall outside anything seen in training, and scaling cannot fix that. So we window *returns and scale-free bar statistics*, not raw prices.

> **Status:** `src/features/patterns.py` is not implemented yet. The patterns section below is a clearly marked placeholder, so this notebook currently produces the **baseline (price-only)** dataset and unblocks Stage 03.

## Environment setup (Google Colab)

Mounts the Drive, clones the repo via the `GITHUB_TOKEN` secret, and installs dependencies.

In [ ]:
import os
import sys

from google.colab import drive, userdata

drive.mount('/content/drive')

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER = 'jesseingraham'
REPO_NAME = 'comp-653-stock-prediction-i'
REPO_PATH = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_PATH):
    !git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git

os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

!pip install -r requirements.txt

## Imports & windowing parameters

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from config import DRIVE_DATA_PATH
from src.data.cleaner import CLEAN_FILENAME
from src.features.base_features import make_base_features
from src.features.windowing import (
    AUGMENTED_WINDOWS_FILENAME,
    BASELINE_WINDOWS_FILENAME,
    make_windows,
    save_windows,
)

# This notebook owns the windowing parameters; the modules stay agnostic.
WINDOW_SIZE = 30      # look-back bars per window
HORIZON = 1           # future steps to predict (y shape: (n, HORIZON))
STRIDE = 1            # every bar starts a window
RETURN_KIND = 'log'   # target = per-step log returns of Close
VOLATILITY_WINDOWS = (5, 20)

print(f'window={WINDOW_SIZE} horizon={HORIZON} stride={STRIDE} returns={RETURN_KIND}')

## 1. Load the cleaned dataset

Produced by Stage 01 (`01_data_collection`) and persisted to the Drive `data/` folder.

In [ ]:
clean = pd.read_parquet(os.path.join(DRIVE_DATA_PATH, CLEAN_FILENAME))

print(f'Loaded {len(clean):,} rows | {clean.index.min().date()} -> {clean.index.max().date()}')
clean.head()

## 2. Derive stationary base features

`log_return`, `high_low_range`, `open_close_change`, `log_volume_change`, and rolling `volatility_*`. Leading rows are NaN by construction (the windowing step drops any window containing NaN).

In [ ]:
features, base_columns = make_base_features(
    clean, volatility_windows=VOLATILITY_WINDOWS
)

print('Base feature columns:', base_columns)
features[base_columns].describe().T

In [ ]:
fig, axes = plt.subplots(
    len(base_columns), 1, figsize=(12, 2.0 * len(base_columns)), sharex=True
)
for ax, col in zip(axes, base_columns):
    ax.plot(features.index, features[col], linewidth=0.6)
    ax.set_ylabel(col, fontsize=8)
    ax.grid(True, alpha=0.3)
axes[0].set_title('Stationary base features')
axes[-1].set_xlabel('Date')
fig.tight_layout()
plt.show()

## 3. Chart pattern features — ⚠️ PLACEHOLDER

**`src/features/patterns.py` is not implemented yet.** When it lands, this section should call it to append pattern columns (e.g. one-hot or confidence scores for detected patterns) to `features`, and set `PATTERNS_AVAILABLE = True`.

Until then `pattern_columns` stays empty and only the **baseline** dataset is written. We deliberately do *not* write an augmented file that is identical to the baseline — that would make the Stage 04 comparison meaningless.

In [ ]:
PATTERNS_AVAILABLE = False  # flip to True once patterns.py exists
pattern_columns: list[str] = []

if PATTERNS_AVAILABLE:
    # TODO: replace with the real patterns API, e.g.
    #   from src.features.patterns import add_pattern_features
    #   features, pattern_columns = add_pattern_features(features)
    raise NotImplementedError('src/features/patterns.py is not ready yet.')

print(f'Pattern features: {len(pattern_columns)} (available={PATTERNS_AVAILABLE})')

## 4. Build and save the baseline windows

Windows the price-only features. Targets are the next `HORIZON` log returns of `Close`. Features are written **unscaled** — the trainer fits scalers per walk-forward fold so cross-validation stays leakage-free.

In [ ]:
X, y, dates = make_windows(
    features,
    feature_columns=base_columns,
    target_column='Close',
    window_size=WINDOW_SIZE,
    horizon=HORIZON,
    stride=STRIDE,
    return_kind=RETURN_KIND,
)

baseline_path = save_windows(X, y, dates, BASELINE_WINDOWS_FILENAME)
print(f'X {X.shape} | y {y.shape} | dates {dates.shape}')
print('Saved ->', baseline_path)

## 5. Build and save the augmented windows

Identical call, with the pattern columns appended to the feature list — so the two datasets differ *only* by those features. Skipped until `patterns.py` exists.

In [ ]:
if PATTERNS_AVAILABLE:
    X_aug, y_aug, dates_aug = make_windows(
        features,
        feature_columns=base_columns + pattern_columns,
        target_column='Close',
        window_size=WINDOW_SIZE,
        horizon=HORIZON,
        stride=STRIDE,
        return_kind=RETURN_KIND,
    )
    augmented_path = save_windows(
        X_aug, y_aug, dates_aug, AUGMENTED_WINDOWS_FILENAME
    )
    print(f'X {X_aug.shape} | y {y_aug.shape}')
    print('Saved ->', augmented_path)
else:
    print('Skipped: patterns.py not implemented, so no augmented dataset.')
    print('Stage 03 (baseline) can run now; Stage 04 waits on patterns.')

## 6. Verify the saved dataset

Reload from the Drive and sanity-check shapes, finiteness, and the target distribution.

In [ ]:
with np.load(os.path.join(DRIVE_DATA_PATH, BASELINE_WINDOWS_FILENAME),
             allow_pickle=True) as loaded:
    X_r, y_r, dates_r = loaded['X'], loaded['y'], loaded['dates']

print('X:', X_r.shape, '| y:', y_r.shape, '| dates:', dates_r.shape)
print('all finite:', np.isfinite(X_r).all() and np.isfinite(y_r).all())
print('target window:', pd.to_datetime(dates_r.min()).date(), '->',
      pd.to_datetime(dates_r.max()).date())
print(f'target mean {y_r.mean():.6f} | std {y_r.std():.6f} | up-days {(y_r > 0).mean():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_r.ravel(), bins=80, color='#1f77b4')
axes[0].set_title('Target log-return distribution')
axes[0].set_xlabel('log return')
axes[1].plot(pd.to_datetime(dates_r), y_r[:, 0], linewidth=0.5,
             color='#1f77b4')
axes[1].set_title('Target log return over time')
axes[1].set_xlabel('Date')
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Summary

The **baseline** windowed dataset is saved to the Shared Drive and Stage 03 can now be run.

**Next:** once `src/features/patterns.py` lands, implement the section 3 placeholder, set `PATTERNS_AVAILABLE = True`, and re-run to emit the augmented dataset for Stage 04.